# How Models Think: Embeddings vs LLMs

**AI-2 Session 2.1 — Opening Demo (20 minutes)**

---

Before you create your first embedding, you need to see something. Let's start with a simple question: **how would you search a collection of documents?**

For the past two weeks, you've been extracting structured data from the Northbrook Partners corpus. Now imagine a user asks a question and you need to find the right document to answer it. The simplest approach? Search for keywords.

---

## The Trap: Let's Try Keyword Search

We have 15 Northbrook Partners documents — memos, policies, financial reports, meeting notes. Let's search them the old-fashioned way: look for a keyword in the text.

In [ ]:
import sys
import os
from pathlib import Path

# Find repo root by walking up until we find pyproject.toml
_here = Path(".").resolve()
_root = _here
while _root != _root.parent:
    if (_root / "pyproject.toml").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))

# Load all Northbrook documents
corpus_dir = _root / "data" / "northbrook"
documents = {}
for f in sorted(corpus_dir.glob("*.md")):
    documents[f.stem] = f.read_text()

print(f"Loaded {len(documents)} Northbrook documents:\n")
for name in documents:
    preview = documents[name][:80].replace("\n", " ").strip()
    print(f"  {name:<40s} ({len(documents[name]):,} chars)")


def keyword_search(keyword: str, docs: dict) -> list:
    """Search documents for a keyword. Returns list of (name, snippet) tuples."""
    results = []
    for name, text in docs.items():
        lower_text = text.lower()
        keyword_lower = keyword.lower()
        pos = lower_text.find(keyword_lower)
        if pos != -1:
            # Extract a snippet around the match
            start = max(0, pos - 40)
            end = min(len(text), pos + len(keyword) + 60)
            snippet = text[start:end].replace("\n", " ").strip()
            if start > 0:
                snippet = "..." + snippet
            if end < len(text):
                snippet = snippet + "..."
            results.append((name, snippet))
    return results


print("\n✓ Corpus loaded. keyword_search() ready.")

In [ ]:
# ── SEARCH 1: "compensation" ──────────────────────────────────
# A user wants to find everything about employee pay and benefits.

print("🔍 SEARCH: \"compensation\"")
print("=" * 60)

results = keyword_search("compensation", documents)

if not results:
    print("\n  ❌ 0 results found.\n")
else:
    for name, snippet in results:
        print(f"\n  📄 {name}")
        print(f"     \"{snippet}\"")

print("\nBut wait — what about these documents?")
print("  • remote_work_policy.md    → $1,500 equipment STIPEND")
print("  • expense_policy.md        → meal REIMBURSEMENT up to $65/day")
print("  • vacation_policy_2025.md  → 20 VACATION days per year")
print("  • hr_committee_2025_01.md  → training BUDGET allocation")
print("\nAll of these are about pay and benefits.")
print("None of them use the word \"compensation.\"")
print("Keyword search missed every single one.")


# ── SEARCH 2: "remote" ───────────────────────────────────────
# A user wants the remote work policy.

print("\n\n🔍 SEARCH: \"remote\"")
print("=" * 60)

results = keyword_search("remote", documents)

for name, snippet in results:
    # Flag whether this is actually about remote work POLICY
    if "remote_work" in name:
        tag = "✅ RELEVANT"
    else:
        tag = "⚠️  WRONG CONTEXT"
    print(f"\n  📄 {name}  [{tag}]")
    print(f"     \"{snippet}\"")

print("\nThe word \"remote\" appears in multiple documents.")
print("But the security memo is about remote ACCESS to servers,")
print("not about employees working from home.")
print("Same word. Different meaning. Keyword search can't tell.")


# ── SEARCH 3: "policy" ───────────────────────────────────────
# A user wants to find all formal company policies.

print("\n\n🔍 SEARCH: \"policy\"")
print("=" * 60)

results = keyword_search("policy", documents)
print(f"\n  {len(results)} documents contain the word \"policy\":\n")
for name, snippet in results:
    print(f"  📄 {name}")

print(f"\n{len(results)} results. Some are actual policies.")
print("Some are memos that MENTION policies. Some are meeting")
print("notes that DISCUSSED policies. A user looking for")
print("\"the remote work policy\" now has to read through all of")
print("them to find what they need.")

---

## The Problem with Keywords

Three searches. Three failures:

| Search | Problem | Result |
|--------|---------|--------|
| `"compensation"` | **Different words, same meaning** — stipend, reimbursement, vacation days | Missed everything |
| `"remote"` | **Same word, different meaning** — remote work vs remote access | False positives |
| `"policy"` | **Too broad** — policies, memos about policies, meeting notes discussing policies | Noise drowns signal |

Keywords match **words**, not **meaning**. What if we could search by meaning instead?

That's what **embeddings** do — they turn text into numbers that capture *what text means*, not just which words it contains. But before you learn how to create embeddings, you need to understand something critical: **the model that searches by meaning is NOT the same model as Claude.** They think differently. And that difference has consequences.

## Setup

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

# Find repo root by walking up until we find pyproject.toml
_here = Path(".").resolve()
_root = _here
while _root != _root.parent:
    if (_root / "pyproject.toml").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))

from dotenv import load_dotenv
load_dotenv()

import anthropic
from src.s2_embeddings.embed import get_embedding, cosine_similarity

# API client for Claude
claude = anthropic.Anthropic()

def ask_claude(question: str, context: str = "") -> str:
    """Ask Claude a question and return the text response."""
    prompt = question if not context else f"{context}\n\n{question}"
    message = claude.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text

def show_similarity_gauge(score: float, label: str = "", expected: str = None):
    """Display a horizontal gradient gauge for a similarity score."""
    fig, ax = plt.subplots(figsize=(7, 1.0))
    gradient = np.linspace(0, 1, 256).reshape(1, -1)
    cmap = mcolors.LinearSegmentedColormap.from_list("sim", ["#2ecc71", "#f1c40f", "#e74c3c"])
    ax.imshow(gradient, aspect="auto", cmap=cmap, extent=[0, 1, 0, 1])
    ax.axvline(x=score, color="black", linewidth=2.5)
    ax.axvline(x=0.70, color="white", linewidth=1.2, linestyle="--", alpha=0.9)
    ax.text(0.70, 1.12, "retrieval\nthreshold", fontsize=7, color="gray",
            ha="center", va="bottom", transform=ax.get_xaxis_transform())
    ax.text(score, -0.25, f"{score:.4f}", fontsize=14, fontweight="bold",
            ha="center", va="top", transform=ax.get_xaxis_transform())
    indicator = ""
    if expected == "similar":
        indicator = " [HIT]" if score >= 0.70 else " [MISS]"
    elif expected == "different":
        indicator = " [MISS — misleading]" if score >= 0.70 else " [HIT]"
    title = label if label else "Similarity"
    ax.set_title(f"{title}{indicator}", fontsize=9, fontweight="bold", loc="left", pad=8)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xticks([0.0, 0.25, 0.5, 0.75, 1.0])
    ax.tick_params(axis="x", labelsize=7)
    plt.tight_layout()
    plt.show()
    plt.close()

print("Setup complete -- Claude API and Voyage AI embedding model ready.")

---

## The Hook: Same Text, Two Models, Different Answers

Let’s send the **exact same pair of sentences** to two different models and see what they say.

## Make Your Prediction

Before we run these tests, commit to a prediction:

> **Two sentences with the same words in different order:**
> *"The manager approved the employee's request"*  
> *"The employee approved the manager's request"*

1. Will the **embedding model** say these are similar or different?
2. Will **Claude** say they're similar or different?

*(Instructor: poll the room before running the next cell)*

In [ ]:
import time

# Two sentences with the same words, different meaning
sentence_a = "The manager approved the employee's request"
sentence_b = "The employee approved the manager's request"

print("=" * 65)
print("  SENTENCE A:", sentence_a)
print("  SENTENCE B:", sentence_b)
print("=" * 65)

# --- What does the EMBEDDING MODEL think? ---
print("\nEMBEDDING MODEL (Voyage AI)")
print("-" * 40)
t0 = time.time()
vec_a = get_embedding(sentence_a)
vec_b = get_embedding(sentence_b)
sim_score = cosine_similarity(vec_a, vec_b)
embed_time = time.time() - t0
print(f"Cosine similarity: {sim_score:.4f}")

show_similarity_gauge(sim_score, label="Embedding: manager/employee word swap", expected="different")

# --- What does the LLM think? ---
print("\nLARGE LANGUAGE MODEL (Claude)")
print("-" * 40)
t0 = time.time()
claude_response = ask_claude(
    "Compare these two sentences. Do they mean the same thing? Be specific about WHO is doing WHAT.\n\n"
    f'Sentence A: "{sentence_a}"\n'
    f'Sentence B: "{sentence_b}"'
)
claude_time = time.time() - t0
print(claude_response)

# --- Claude's verdict bar ---
fig, ax = plt.subplots(figsize=(7, 0.5))
ax.barh(0, 1, color='#2ecc71', height=0.6)
ax.text(0.5, 0, 'CORRECTLY IDENTIFIED: Different meanings, reversed roles',
        ha='center', va='center', fontsize=9, fontweight='bold', color='white')
ax.set_xlim(0, 1)
ax.set_ylim(-0.5, 0.5)
ax.axis('off')
ax.set_title("Claude's Verdict", fontsize=9, fontweight='bold', loc='left')
plt.tight_layout()
plt.show()
plt.close()

print(f"\nTiming -- Embedding: {embed_time:.2f}s | Claude: {claude_time:.2f}s")
print(f"   Embedding is ~{claude_time/embed_time:.0f}x faster -- but misses the meaning.")

**What just happened?** The embedding model thinks these are nearly identical. Claude immediately sees that the *actors are reversed* — completely different meaning. Same words, different order, and only one model notices.

Let’s see two more examples where these models disagree.

In [ ]:
# --- PUNCH 2: Negation ---
print("=" * 65)
print("  NEGATION TEST")
print("=" * 65)

neg_a = "The treatment improved patient outcomes"
neg_b = "The treatment did NOT improve patient outcomes"

print(f'\n  A: "{neg_a}"')
print(f'  B: "{neg_b}"')

vec_a = get_embedding(neg_a)
vec_b = get_embedding(neg_b)
neg_score = cosine_similarity(vec_a, vec_b)
print(f"Cosine similarity: {neg_score:.4f}")

show_similarity_gauge(neg_score, label="Embedding: statement vs its negation", expected="different")

print("\nClaude's take:")
claude_neg = ask_claude(
    f'Do these mean the same thing?\nA: "{neg_a}"\nB: "{neg_b}"\nAnswer in 1-2 sentences.'
)
print(claude_neg)

# --- PUNCH 3: Word Scramble ---
import random

print("\n" + "=" * 65)
print("  WORD SCRAMBLE TEST")
print("=" * 65)

original = "The quarterly budget report shows a significant increase in revenue"
words = original.split()
random.seed(42)  # reproducible
scrambled_words = words.copy()
random.shuffle(scrambled_words)
scrambled = " ".join(scrambled_words)

print(f'\n  Original:  "{original}"')
print(f'  Scrambled: "{scrambled}"')

vec_a = get_embedding(original)
vec_b = get_embedding(scrambled)
scramble_score = cosine_similarity(vec_a, vec_b)
print(f"Cosine similarity: {scramble_score:.4f}")

show_similarity_gauge(scramble_score, label="Embedding: coherent vs scrambled sentence", expected="different")

print("\nClaude's take:")
claude_scramble = ask_claude(
    f'Compare these two texts. Is the second one meaningful?\nA: "{original}"\nB: "{scrambled}"\nAnswer in 1-2 sentences.'
)
print(claude_scramble)

In [ ]:
# ── SCOREBOARD: Embeddings vs Claude ──────────────────────────
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')

col_labels = ['Test', 'Similarity', 'Embedding Says', 'Claude Says']
row_data = [
    ['Word Swap',  f'{sim_score:.2f}', '"Nearly identical"', '"Opposite meaning"'],
    ['Negation',   f'{neg_score:.2f}', '"Very similar"',     '"Contradictory"'],
    ['Scramble',   f'{scramble_score:.2f}', '"Similar"',     '"Nonsensical"'],
]

table = ax.table(
    cellText=row_data,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    colWidths=[0.18, 0.15, 0.30, 0.30],
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.8)

# Style header row
for j in range(len(col_labels)):
    cell = table[0, j]
    cell.set_facecolor('#2c3e50')
    cell.set_text_props(color='white', fontweight='bold')

# Color code data rows: green for Claude (correct), red for Embedding (misleading)
for i in range(1, 4):
    # Similarity column — red (misleading high score)
    table[i, 1].set_facecolor('#fadbd8')
    # Embedding Says — red
    table[i, 2].set_facecolor('#fadbd8')
    table[i, 2].set_text_props(color='#c0392b')
    # Claude Says — green
    table[i, 3].set_facecolor('#d5f5e3')
    table[i, 3].set_text_props(color='#1e8449')

ax.set_title('Scoreboard: Embeddings vs Claude', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()
plt.close()

print("\n3 for 3 -- embeddings missed structural meaning every time.")
print("They see the WORDS. Claude sees the MEANING.")

## But Wait -- Embeddings Are Not Broken

Remember the keyword search failure at the top? Searching for "compensation" missed documents about stipends, reimbursement, and vacation days. Let's see what embeddings do with that exact same problem.

In [ ]:
# ── WHERE EMBEDDINGS WIN: Same meaning, different words ──────
# Remember keyword search #1? "compensation" missed documents about
# stipends, reimbursement, and vacation. Let's see what embeddings do.

query = "employee compensation and benefits"
doc_snippets = {
    "stipend":       "a $1,500 equipment stipend for home office setup",
    "reimbursement": "meal reimbursement up to $65 per day during travel",
    "vacation":      "employees receive 20 vacation days per year",
    "unrelated":     "the board approved the Q3 security audit timeline",
}

print("=" * 65)
print("  WHERE EMBEDDINGS WIN")
print("=" * 65)
print(f'\n  Query: "{query}"\n')

query_vec = get_embedding(query)

for label, snippet in doc_snippets.items():
    vec = get_embedding(snippet)
    score = cosine_similarity(query_vec, vec)
    marker = "<<< keyword search MISSED this" if label != "unrelated" else "<<< correctly lower"
    print(f"  {score:.4f}  \"{snippet[:55]}...\"  {marker}")

print("\n" + "-" * 65)
print("Keyword search for \"compensation\" found ZERO of these.")
print("Embeddings rank them all above the unrelated document.")
print("Same meaning, different words — embeddings see right through it.")

---

## Why They Disagree: Compression vs Comprehension

These two model types process text in fundamentally different ways.

### Embedding Model (Voyage AI) — The Compressor

The embedding model reads your **entire input at once** and compresses it into a single fixed-length vector (a list of 1,024 numbers). Think of it like GPS coordinates for meaning — nearby coordinates mean similar topics.

**What it keeps:** Which words are present, general topic, overall sentiment  
**What it loses:** Word order, negation, who-did-what-to-whom relationships

```
"The manager approved the employee's request"
    → [0.23, -0.81, 0.45, 0.12, ...] (1,024 numbers)

"The employee approved the manager's request"  
    → [0.21, -0.79, 0.47, 0.11, ...] (1,024 numbers)
    
    ← Nearly identical! Same words = similar coordinates.
```

### Large Language Model (Claude) — The Comprehender

Claude processes tokens **one at a time, left to right**, using attention to track how each word relates to every other word. It builds a rich understanding of structure, roles, and relationships.

```
"The manager approved the employee's request"
      ↓         ↓         ↓
  [subject]  [action]  [object's possession]
  WHO        DID WHAT  TO WHOM
  
"The employee approved the manager's request"  
      ↓          ↓         ↓
  [subject]   [action]  [object's possession]
  DIFFERENT   SAME      DIFFERENT
  WHO         ACTION    TO WHOM
```

### The Map vs The Navigator

**Embedding = GPS coordinates.** "123 Main St" and "124 Main St" are almost identical coordinates. But one might be a bakery and the other a bank. The coordinates tell you they’re close, not what’s inside.

**LLM = A person who reads the addresses, walks to both buildings, and tells you what they found.** They understand context, purpose, and nuance.

Both are useful. But they see different things.

---

## Try It Yourself: The Disagreement Detector

Type any two sentences below. The notebook will show you what the embedding model thinks (similarity score) AND what Claude thinks (semantic analysis). Look for cases where they **disagree**.

**Tip:** Try swapping roles, adding "not", or saying the same thing in very different words.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

text_a_input = widgets.Text(
    value="The company fired the CEO",
    placeholder="Type sentence A...",
    description="Sentence A:",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "85px"},
)

text_b_input = widgets.Text(
    value="The CEO fired the company",
    placeholder="Type sentence B...",
    description="Sentence B:",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "85px"},
)

compare_button = widgets.Button(
    description="Compare",
    button_style="primary",
    layout=widgets.Layout(width="150px"),
)

output_area = widgets.Output()

# Pre-loaded tricky examples
examples = [
    ("The company fired the CEO", "The CEO fired the company"),
    ("This restaurant is not expensive", "This restaurant is expensive"),
    ("Revenue increased by 50%", "Revenue decreased by 50%"),
    ("What are the rules for working from home?", "Northbrook Partners maintains a flexible remote work arrangement"),
    ("Implement RBAC with OIDC SSO", "Set up role-based access control with single sign-on"),
]
example_idx = [0]

load_example_btn = widgets.Button(
    description="Load Tricky Example",
    button_style="info",
    layout=widgets.Layout(width="180px"),
)

def on_load_example(b):
    idx = example_idx[0] % len(examples)
    text_a_input.value = examples[idx][0]
    text_b_input.value = examples[idx][1]
    example_idx[0] += 1

load_example_btn.on_click(on_load_example)

def on_compare(b):
    a = text_a_input.value.strip()
    b_text = text_b_input.value.strip()
    if not a or not b_text:
        return
    
    with output_area:
        clear_output(wait=True)
        
        # Embedding comparison
        print("EMBEDDING MODEL")
        print("-" * 40)
        vec_a = get_embedding(a)
        vec_b = get_embedding(b_text)
        score = cosine_similarity(vec_a, vec_b)
        
        show_similarity_gauge(score, label=f"Embedding similarity")
        
        emb_verdict = "SIMILAR" if score >= 0.70 else "DIFFERENT"
        
        # Claude comparison
        print("\nCLAUDE")
        print("-" * 40)
        claude_analysis = ask_claude(
            f'Compare the meanings of these two sentences in 2-3 sentences. '
            f'Are they saying the same thing or different things? Be specific.\n\n'
            f'A: "{a}"\nB: "{b_text}"'
        )
        print(claude_analysis)
        
        # Agreement check
        print("\n" + "=" * 50)
        if score >= 0.70:
            print(f"  Embedding says: SIMILAR ({score:.4f})")
            print(f"  Does Claude agree? Read above and decide.")
            print(f"  If not -- you found a retrieval blind spot.")
        else:
            print(f"  Embedding says: DIFFERENT ({score:.4f})")
            print(f"  Does Claude agree? Read above and decide.")
        print("=" * 50)

compare_button.on_click(on_compare)

display(
    widgets.VBox([
        text_a_input,
        text_b_input,
        widgets.HBox([compare_button, load_example_btn]),
        output_area
    ])
)

---

## Why This Matters: The RAG Blind Spot

In **Retrieval-Augmented Generation (RAG)** — which you’ll build starting in Session 3.1 — these two model types work together:

1. **Embedding model** searches your documents (fast, cheap, but blind to structure)
2. **LLM** reads the retrieved documents and generates an answer (slow, expensive, but understands nuance)

The danger:

```
User asks: "Did the manager approve the employee's request?"

Your document says: "The employee approved the manager's request"

Step 1 — Embedding retrieval: "These are 95% similar!" ← RETRIEVES IT
Step 2 — Claude reads the chunk: "Yes, the request was approved." ← WRONG ACTOR
Step 3 — User gets a confident, wrong answer
```

**The embedding retrieved the wrong chunk. Claude faithfully processed it. The answer is confidently wrong.**

This is not a bug — it’s how embeddings work. They compress text into coordinates, and that compression loses structural information. The rest of this session teaches you how embeddings work, what they’re good at, and how to choose the right model. The `embedding_fails.ipynb` notebook dives deep into exactly which types of information get lost.

> **The skill is not avoiding embeddings** — they’re the best tool for semantic search. The skill is knowing *where they break* so you can engineer around it.

---

*Now let’s learn how to create embeddings and what makes different models better or worse. Open `session_2_1.ipynb`.*